## **Installing Snorkel**

In [ ]:
!pip install snorkel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 578.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
import pandas as pd
import numpy as np
import re
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from snorkel.labeling.model import LabelModel


In [ ]:
ABSTAIN = -1
POSITIVE = 1
NEGATIVE = 0

# Load dataset
df = pd.read_csv("movie.csv", on_bad_lines="skip", quoting=3, encoding="utf-8")
df = df.dropna(subset=["text"])
df.head()


,,text,label
"""I grew up (b. 1965) watching and loving the Thunderbirds. All my mates at school watched. We played """"Thunderbirds"""" before school",during lunch and after school. We all wanted to be Virgil or Scott. No one wanted to be Alan. Counting down from 5 became an art form. I took my children to see the movie hoping they would get a glimpse of what I loved as a child. How bitterly disappointing. The only high point was the snappy theme tune. Not that it could compare with the original score of the Thunderbirds. Thankfully early Saturday mornings one television channel still plays reruns of the series Gerry Anderson and his wife created. Jonatha Frakes should hand in his directors chair,his version was completely hopeless. A waste ...,0.0
"""Even though I have great interest in Biblical movies",I was bored to death every minute of the movie. Everything is bad. The movie is too long,the acting is most of the time a Joke and the...,0.0
"""If you want a fun romp with loads of subtle humor",then you will enjoy this flick.<br /><br />I don't understand why anyone wouldn't enjoy this one. Take it for what it is: a vehicle for Dennis Hopper to mess with your head and make you laugh. It ain't Shakespeare,but it is well done. Ericka Eleniak is absolu...,1.0
"""SUcks. That's all I got to say about this sorry excuse for a film. Sucks. Sucks. Sucks. I mean","what the hell were they thinking? The idiots involved should never be allowed to make another films. The acting was so bad that it even failed to entertain on a bad level. The attempt at a """"lesbian scene"""" was sad. I felt so bad for the ladies involved. This movie sucks! Sucks! Sucks!<br /><br />I heard rumors of a sequel.<br /><br />God<br /><br />Help<br /><br />Us<br /><br />All""",0,NaN
"""I love this movie ! I think I've seen it 5 times already (it was quite a success in France and they often play it on TV). Ok","it's a thriller and there is great tension. But mostly (and specifically in the second part) it is absolutely hilarious ! And very original. The directing and photography are just splendid.""",1,NaN


In [ ]:
@labeling_function()
def lf_positive_words(row):
    text = str(row.text)  # Ensure it's a string
    positive_words = ["love", "great", "excellent", "fantastic", "amazing", "best", "perfect"]
    return POSITIVE if any(word in text.lower() for word in positive_words) else ABSTAIN

@labeling_function()
def lf_negative_words(row):
    text = str(row.text)  # Ensure it's a string
    negative_words = ["worst", "bad", "horrible", "terrible", "disappointed", "poor", "broken"]
    return NEGATIVE if any(word in text.lower() for word in negative_words) else ABSTAIN

@labeling_function()
def lf_negation(row):
    text = str(row.text)  # Ensure it's a string
    neg_pattern = r"(not\s+good|never\s+buy|no\s+recommendation)"
    return NEGATIVE if re.search(neg_pattern, text.lower()) else ABSTAIN


In [ ]:
lfs = [lf_positive_words, lf_negative_words, lf_negation]

applier = PandasLFApplier(lfs=lfs)
label_matrix = applier.apply(df[["text"]])  # Ensure only the 'text' column is passed


100%|██████████| 4419/4419 [00:00<00:00, 13935.26it/s]


In [ ]:
LFAnalysis(label_matrix, lfs).lf_summary()


,j,Polarity,Coverage,Overlaps,Conflicts
lf_positive_words,0,[1],0.135325,0.016746,0.016746
lf_negative_words,1,[0],0.076262,0.016746,0.016746
lf_negation,2,[0],0.000226,0.000000,0.000000


from matplotlib import pyplot as plt
_df_0['j'].plot(kind='hist', bins=20, title='j')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['Coverage'].plot(kind='hist', bins=20, title='Coverage')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2['Overlaps'].plot(kind='hist', bins=20, title='Overlaps')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_3['Conflicts'].plot(kind='hist', bins=20, title='Conflicts')
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['j']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'j'}, axis=1)
              .sort_values('j', ascending=True))
  xs = counted['j']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_10.sort_values('j', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('j')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
_df_11['j'].plot(kind='line', figsize=(8, 4), title='j')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_12['Coverage'].plot(kind='line', figsize=(8, 4), title='Coverage')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_13['Overlaps'].plot(kind='line', figsize=(8, 4), title='Overlaps')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_14['Conflicts'].plot(kind='line', figsize=(8, 4), title='Conflicts')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
label_model = LabelModel(cardinality=2, verbose=True)
label_model.fit(label_matrix, n_epochs=500, log_freq=100, seed=42)


100%|██████████| 500/500 [00:00<00:00, 773.19epoch/s]


In [ ]:
df["snorkel_label"] = label_model.predict(label_matrix)


In [ ]:
df["snorkel_label"].value_counts()


,count
snorkel_label,
-1,3557
1,861
0,1


In [ ]:
df.to_csv("snorkel_weakly_labeled.csv", index=False)
print("✅ Snorkel labeling complete! Labeled dataset saved in the current directory.")


✅ Snorkel labeling complete! Labeled dataset saved in the current directory.
